# Random Forest & Ensemble Methods Interview Questions & Answers

**Target roles:** ML Engineer · Data Scientist · Applied Scientist · MLOps Engineer  
**Difficulty levels:** 🟢 Foundational · 🟡 Intermediate · 🔴 Advanced · 💀 Trick

---

## Introduction

Random Forest and ensemble methods form one of the most reliable toolkits in applied ML. Despite the dominance of gradient boosting frameworks (XGBoost, LightGBM, CatBoost) and deep learning, Random Forests remain a staple baseline and a favourite interview topic — because they elegantly demonstrate core statistical principles: the **bias-variance tradeoff**, the **wisdom of crowds**, and **model regularisation through randomness**.

This notebook covers every question that regularly appears in ML interviews at top-tier tech companies, broken down as follows:

| Section | # Questions |
|---|---|
| Random Forest Fundamentals | 12 |
| Ensemble Methods: Bagging, Boosting, Stacking | 10 |
| Trick Questions | 6 |
| Advanced Questions | 6 |
| Code Demonstrations | 4 cells |

**How to use:** Try to answer each question yourself before reading the answer. Cover the answer block, think for 30 seconds, then compare. The act of retrieval — even failed retrieval — dramatically improves retention.

---
## Section 1: Random Forest Fundamentals

---

### Q1 🟢 What is a Random Forest, and how does it differ from a single decision tree?

**Answer:**  
A Random Forest is an ensemble of decision trees, where each tree is trained on a **bootstrap sample** of the training data, and each split considers only a **random subset of features**. Predictions are aggregated by majority vote (classification) or averaging (regression). A single decision tree has high variance — small changes in training data produce very different trees — and tends to overfit. A Random Forest reduces this variance through averaging: if each tree's error is partially independent, their average error shrinks, while the systematic (bias) component stays roughly the same.

$$
\text{Var}\left(\frac{1}{B}\sum_{b=1}^{B} T_b\right) = \rho \sigma^2 + \frac{1-\rho}{B}\sigma^2
$$

where $\rho$ is the pairwise correlation between trees, $\sigma^2$ is each tree's variance, and $B$ is the number of trees. As $B \to \infty$, the second term vanishes, leaving $\rho\sigma^2$ — the irreducible component due to tree correlation.

---

### Q2 🟢 Explain bagging (bootstrap aggregation). What problem does it solve?

**Answer:**  
Bagging (Breiman, 1996) trains $B$ models on $B$ **bootstrap samples** — each sample drawn with replacement from the original training set, so each bootstrap dataset has the same size as the original but contains ~63.2% unique rows on average (the rest are duplicates). The final prediction averages (or votes across) all $B$ models. Bagging addresses **high variance** in unstable learners like deep decision trees: averaging multiple noisy-but-unbiased estimators produces a smoother estimator with lower variance and comparable bias. It does *not* help with high-bias models — averaging many bad models still gives a bad model.

$$
P(\text{row } i \text{ not selected}) = \left(1 - \frac{1}{n}\right)^n \xrightarrow{n\to\infty} e^{-1} \approx 0.368
$$

---

### Q3 🟢 What is the "random" in Random Forest? Why is it important?

**Answer:**  
There are **two sources of randomness**: (1) each tree trains on a bootstrap sample of rows, and (2) at every split, only a random subset of $m$ features is considered — not all $p$ features. This *feature subsampling* is the key innovation over plain bagged trees. Without it, if one feature is very strong, all trees will use it near the root and become highly correlated, limiting the variance reduction. By restricting the feature candidates at each split, Random Forest forces trees to use different features, **decorrelating** them and pushing $\rho$ in the formula above toward zero. Lower $\rho$ means more variance reduction from averaging.

---

### Q4 🟡 How many features should be considered at each split? What are the rules of thumb?

**Answer:**  

| Task | Default $m$ | Rationale |
|---|---|---|
| Classification | $m = \lfloor\sqrt{p}\rfloor$ | Breiman's empirical recommendation |
| Regression | $m = \lfloor p/3 \rfloor$ | Avoids over-reliance on one feature |

These defaults work well in practice but are hyperparameters worth tuning (`max_features` in scikit-learn). Setting $m = p$ reduces to a plain bagged tree. Setting $m = 1$ maximises diversity at the cost of expressiveness per tree. The sweet spot balances tree strength (individual accuracy) against tree diversity (decorrelation).

---

### Q5 🟡 What is Out-of-Bag (OOB) error and how is it calculated?

**Answer:**  
Since each tree is trained on ~63.2% of the data, the remaining ~36.8% of rows are **out-of-bag** for that tree — they were never seen during its training. For each training example $x_i$, we collect predictions only from trees for which $x_i$ was OOB, then aggregate those predictions. The OOB error is the error of this aggregated OOB prediction across all training examples. This gives a **free, nearly unbiased estimate of generalisation error** without needing a held-out validation set — making it especially useful when data is scarce. OOB error converges to leave-one-out cross-validation error asymptotically.

---

### Q6 🟡 How does Random Forest reduce variance but not bias?

**Answer:**  
Each individual tree in a Random Forest is grown **fully** (no pruning, deep splits) — this makes each tree a low-bias, high-variance estimator. Because each tree sees nearly the full signal, the ensemble has the same low bias. Averaging $B$ (partially) independent estimators reduces variance from $\sigma^2$ toward $\rho \sigma^2$ without touching the bias component. If instead you used shallow trees (high-bias estimators), bagging would not help much because it cannot reduce bias. This is why Random Forest consistently beats pruned single trees but not always boosted ensembles that *do* reduce bias iteratively.

---

### Q7 🟡 What is feature importance in Random Forest? Describe the two main types.

**Answer:**  

**1. Mean Decrease in Impurity (MDI) / Gini importance:**  
For each feature, sum the total reduction in Gini (or entropy) it achieves across all splits in all trees, weighted by the number of samples passing through each node. Fast to compute (free at training time). **Bias:** strongly favours high-cardinality and continuous features because they offer more split opportunities.

**2. Permutation importance (Mean Decrease in Accuracy):**  
After training, for each feature: randomly shuffle its values in the OOB set, measure the increase in error, and average over all trees. A feature that matters a lot will cause a large accuracy drop when permuted. More reliable than MDI, handles cardinality bias, but is slower (requires re-scoring after each permutation). Preferred for model interpretation.

---

### Q8 🟡 What are the key hyperparameters of Random Forest and how do they affect performance?

**Answer:**  

| Hyperparameter | Effect |
|---|---|
| `n_estimators` | More trees → lower variance, diminishing returns after ~200–500. Never hurts, only slows training. |
| `max_features` | Fewer features → more diverse trees, less correlated, but each tree weaker. Tune via CV. |
| `max_depth` | Shallow trees → higher bias, lower variance; deeper trees → lower bias, higher variance. `None` = fully grown. |
| `min_samples_split` / `min_samples_leaf` | Larger values regularise the model (prevent overfitting), act like tree pruning. |
| `max_samples` | Bootstrap sample size. Smaller → more diversity but weaker individual trees. |
| `class_weight` | Handles class imbalance by upweighting minority class samples during split scoring. |

The most impactful tuning knobs are `n_estimators`, `max_features`, and `max_depth`.

---

### Q9 🟡 How does Random Forest handle missing values?

**Answer:**  
Scikit-learn's `RandomForestClassifier` does **not** natively handle missing values — you must impute before fitting. However, Breiman's original implementation and some other libraries (e.g., R's `randomForest`) use a **proximity-based imputation**: the forest is first trained with median/mode imputation, then the proximity matrix (fraction of times two samples end up in the same leaf) is used to re-impute missing values as a weighted average of non-missing values, weighted by proximity. This process can be iterated. In practice for scikit-learn, common strategies are median/mean imputation, or using a tree-native library like XGBoost that handles missings natively during splitting.

---

### Q10 🟡 How does Random Forest handle class imbalance?

**Answer:**  
Several strategies exist: (1) **`class_weight='balanced'`** — scikit-learn automatically weights each class inversely proportional to its frequency ($w_c = n / (k \cdot n_c)$), influencing the impurity calculation at splits. (2) **Balanced bagging** — each bootstrap sample is drawn with equal representation from each class (stratified sampling). (3) **Undersampling the majority class** when building each bootstrap sample (used in `BalancedRandomForestClassifier` in `imbalanced-learn`). (4) Post-hoc **threshold tuning** using the OOB predicted probabilities. Class weighting is the easiest first step; balanced bagging often gives better recall on the minority class.

---

### Q11 🟡 Explain Gini impurity vs Information Gain (entropy) as split criteria. Which should you prefer?

**Answer:**  

$$
\text{Gini}(t) = 1 - \sum_{k=1}^{K} p_k^2 \qquad \text{Entropy}(t) = -\sum_{k=1}^{K} p_k \log_2 p_k
$$

Both measure node impurity; both equal zero for a pure node and are maximised for uniform distributions. Gini is slightly faster to compute (no logarithm). Empirically they produce nearly identical trees and performance. Entropy can sometimes yield slightly more balanced trees because $\log$ grows faster near 0, penalising extreme class dominance more. In practice, the choice rarely matters — the much larger driver of performance is the other hyperparameters. The default `criterion='gini'` in scikit-learn is fine for most use cases.

---

### Q12 🔴 What is the relationship between tree depth, number of trees, and overfitting in Random Forest?

**Answer:**  
A fully grown tree has near-zero training error (low bias, high variance). Averaging many such trees reduces variance, so a large forest of fully grown trees is typically well-regularised and does not overfit significantly — empirically, Random Forests are remarkably robust to overfitting. However, with very noisy datasets or very small training sets, too-deep trees can still overfit (each tree memorises noise, and even averaging doesn't fully cancel it). In those cases, constraining `max_depth` or increasing `min_samples_leaf` reduces variance further at the cost of some bias. Increasing `n_estimators` always helps (or at worst plateaus) because more trees can only reduce variance.

---

## Section 2: Ensemble Methods — Bagging, Boosting, and Stacking

---

### Q13 🟢 What is the fundamental difference between bagging and boosting?

**Answer:**  

| | Bagging | Boosting |
|---|---|---|
| Tree training | **Parallel** — independent | **Sequential** — each tree corrects the previous |
| Sample weighting | Uniform bootstrap | Upweight misclassified examples |
| Primary target | Reduce **variance** | Reduce **bias** (and variance) |
| Overfitting risk | Low | Higher — sensitive to noisy labels |
| Example | Random Forest | AdaBoost, Gradient Boosting, XGBoost |

Bagging builds each model independently on a bootstrap sample; boosting builds each model to correct the *residual errors* of the previous ensemble. Because boosting focuses on hard examples, it can reduce bias, but the sequential nature makes it more prone to overfitting noisy data.

---

### Q14 🟡 Explain AdaBoost. How does it work step by step?

**Answer:**  
AdaBoost (Adaptive Boosting, Freund & Schapire 1997) trains a sequence of **weak learners** (typically decision stumps — trees with depth 1) where each learner is trained on a re-weighted dataset:

1. Initialise sample weights: $w_i = 1/n$ for all $i$.
2. For $t = 1, \ldots, T$:
   - Fit weak learner $h_t$ using weights $w_i$.
   - Compute weighted error: $\epsilon_t = \sum_{i: h_t(x_i)\neq y_i} w_i$.
   - Compute learner weight: $\alpha_t = \frac{1}{2}\ln\frac{1-\epsilon_t}{\epsilon_t}$.
   - Update: increase weight for misclassified examples ($w_i \leftarrow w_i e^{\alpha_t}$), decrease for correct.
   - Renormalise weights.
3. Final prediction: $H(x) = \text{sign}\left(\sum_t \alpha_t h_t(x)\right)$.

AdaBoost has an exponential loss function perspective; it is very sensitive to outliers (which get high weights and dominate training).

---

### Q15 🟡 Explain Gradient Boosting. How does it differ from AdaBoost?

**Answer:**  
Gradient Boosting (Friedman, 2001) generalises boosting to arbitrary differentiable loss functions. Instead of reweighting samples, each new tree fits the **negative gradient** (pseudo-residuals) of the loss with respect to the current ensemble's predictions:

$$
r_{im} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}}
$$

For MSE loss, pseudo-residuals are the raw residuals $y_i - \hat{y}_i$. For log-loss (binary classification), they relate to the probability errors. AdaBoost is a special case of gradient boosting with exponential loss. Key differences: AdaBoost uses hard sample reweighting and decision stumps; GBM uses shallow trees ($\text{depth}\approx 3$–8) and gradient descent in function space, supporting any loss function and learning rate.

---

### Q16 🟡 What is stacking (stacked generalisation)? How does it differ from blending?

**Answer:**  
**Stacking** trains a **meta-learner** (level-1 model) on the out-of-fold predictions of multiple base models (level-0 models). To avoid leakage, base models are trained using $k$-fold cross-validation: each fold's predictions are made by a model that has *never seen that fold*. The full training set's OOF predictions form the meta-learner's input features. **Blending** is a simpler variant: the training data is split into a *hold-out blend set* once, base models are trained on the remaining data, and the meta-learner trains on the hold-out blend set predictions. Stacking uses data more efficiently (via CV) but is more complex and computationally expensive. Blending is faster but wastes training data.

---

### Q17 🟡 What is Extra Trees (Extremely Randomized Trees)? How does it differ from Random Forest?

**Answer:**  
Extra Trees (Geurts et al., 2006) adds a **third source of randomness**: instead of finding the *best* split threshold for each candidate feature, it draws the threshold **randomly** from the feature's range. This makes individual trees weaker but even more decorrelated, often reducing variance further. Extra Trees is typically **faster to train** than Random Forest (no need to search for optimal thresholds) and sometimes achieves better generalisation on high-dimensional data. It does *not* use bootstrap sampling by default (trains on the full dataset), relying solely on random feature and threshold selection for diversity. In practice, the performance difference between RF and Extra Trees is dataset-dependent.

---

### Q18 🔴 What is XGBoost and what innovations does it add over vanilla Gradient Boosting?

**Answer:**  
XGBoost (Chen & Guestrin, 2016) adds several engineering and algorithmic improvements over vanilla GBM:

1. **Regularisation:** L1 ($\alpha$) and L2 ($\lambda$) regularisation on leaf weights in the loss function, reducing overfitting.
2. **Second-order gradients:** Uses both gradient ($g_i$) and Hessian ($h_i$) of the loss for more precise leaf weight computation: $w_j^* = -\frac{\sum_{i\in j} g_i}{\sum_{i\in j} h_i + \lambda}$.
3. **Column subsampling:** Like RF's `max_features`, randomly select features per tree or per split.
4. **Sparsity-aware splitting:** Native handling of missing values — learns which branch to send missing values to.
5. **Approximate tree learning:** Weighted quantile sketch for efficient split finding on large datasets.
6. **Cache-aware and out-of-core computation:** Efficient memory layout for speed.

---

### Q19 🟡 When would you choose Random Forest over Gradient Boosting (or XGBoost)?

**Answer:**  

| Situation | Prefer |
|---|---|
| Noisy labels, many outliers | Random Forest (less sensitive) |
| Need fast, parallel training | Random Forest |
| Small dataset, risk of overfitting | Random Forest |
| Want a strong baseline quickly | Random Forest |
| Need maximum predictive accuracy | Gradient Boosting / XGBoost |
| Structured tabular data competition | XGBoost / LightGBM |
| Interpretability via feature importance | Either (similar quality) |

Random Forest is more robust out-of-the-box and easier to tune. Gradient boosting typically achieves better accuracy but requires more careful regularisation tuning and is slower to train sequentially.

---

### Q20 🟡 What is voting ensemble? Describe hard vs soft voting.

**Answer:**  
A **voting ensemble** combines multiple heterogeneous classifiers (e.g., logistic regression + SVM + Random Forest). **Hard voting** takes the majority class predicted by all classifiers: $\hat{y} = \text{mode}(h_1(x), h_2(x), \ldots, h_k(x))$. **Soft voting** averages the predicted class probabilities across classifiers: $\hat{y} = \arg\max_k \frac{1}{K}\sum_{j=1}^{K} P_j(y=k|x)$. Soft voting generally outperforms hard voting because it leverages confidence information — a model that is very confident in class A and another that is uncertain provides more nuance than a raw vote count. Soft voting requires all classifiers to output calibrated probabilities.

---

### Q21 🔴 Explain the bias-variance tradeoff in ensemble methods. How do bagging and boosting each address it?

**Answer:**  
Total expected error decomposes as: $\text{Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$. A single deep decision tree has low bias (flexible enough to fit anything) but high variance (sensitive to training data). **Bagging** averages many such high-variance, low-bias models: variance drops from $\sigma^2$ toward $\rho\sigma^2$ (limited by inter-tree correlation), bias is unchanged. **Boosting** starts with high-bias weak learners (stumps) and iteratively reduces bias by fitting residuals; it also reduces variance because many estimators are averaged. However, boosting can overfit (increase variance on test set) if run too long without regularisation. The two strategies thus attack the tradeoff from opposite directions.

---

### Q22 🔴 Why does tree diversity matter in an ensemble, and how does correlation between trees limit variance reduction?

**Answer:**  
From the variance formula $\text{Var}(\bar{T}) = \rho\sigma^2 + \frac{(1-\rho)\sigma^2}{B}$, the achievable variance floor is $\rho\sigma^2$ regardless of how many trees you add. If all trees are identical (perfectly correlated, $\rho = 1$), averaging does nothing. If all trees are uncorrelated ($\rho = 0$), variance vanishes as $B \to \infty$. Feature subsampling in Random Forest specifically targets $\rho$: by forcing different trees to use different features, it breaks the dominant feature's stranglehold and produces genuinely different partitions of feature space. This is why RF typically beats plain bagged trees despite each individual RF tree being slightly weaker.

---

## Section 3: Trick Questions 💀

*These are designed to catch candidates who have memorised definitions without deep understanding.*

---

### TRICK Q1 💀 "Does adding more trees to a Random Forest always improve performance?"

**Common wrong answer:** "Yes, more trees always improve accuracy."

**Correct answer:**  
Adding trees **never hurts** test performance in theory (the ensemble can only become more stable), but returns **diminish rapidly** and eventually plateau. Once variance is reduced to the floor set by inter-tree correlation ($\rho\sigma^2$), no additional trees help. In practice, after ~200–500 trees, improvements are negligible. The catch is that more trees **always** increase training time and memory linearly, so there is an engineering cost. The correct statement is: more trees reduce variance monotonically until the correlation floor is hit, after which they plateau — they do not continue to improve, and they do not overfit.

---

### TRICK Q2 💀 "Is Random Forest a linear or non-linear model?"

**Common wrong answer:** "Non-linear" (correct but incomplete / said without understanding).

**Correct answer:**  
Random Forest is decidedly **non-linear**. Each decision tree partitions feature space into axis-aligned rectangular regions, producing a step-function prediction surface with sharp boundaries — highly non-linear. The ensemble average smooths these boundaries but remains non-linear. The prediction function cannot be written as $\hat{y} = \mathbf{w}^T \mathbf{x} + b$ globally. This means RF can capture interactions and non-linearities automatically, which is a major strength over linear models. However, the follow-up gotcha: **RF is linear in the leaf indicator representation** — i.e., if you encode which leaf each sample falls into (one-hot), the aggregate is a linear function of those indicators.

---

### TRICK Q3 💀 "Can Random Forest extrapolate beyond the range of training data?"

**Common wrong answer:** "Yes, like any ML model."

**Correct answer:**  
**No.** This is a critical limitation. Decision trees predict by assigning a leaf's average (regression) or majority class (classification), and the leaf boundaries are determined by training data values. For any input outside the range of a training feature, the tree routes it to the most extreme leaf — effectively **clamping** the prediction to the maximum or minimum training value seen. Linear models, polynomial models, and neural networks can extrapolate (though they may extrapolate poorly). Random Forests fundamentally cannot: if your test data includes house sizes larger than any in training, the RF will predict the same price as the largest training house regardless. This makes RF unreliable for forecasting tasks with distribution shift.

---

### TRICK Q4 💀 "Does feature scaling (normalisation/standardisation) matter for Random Forest?"

**Common wrong answer:** "Yes, you should always scale features."

**Correct answer:**  
**No.** Random Forest (like all tree-based models) is **invariant to monotonic transformations of features** — including scaling, normalisation, and log transforms. Decision trees split features at thresholds; doubling all values of a feature just doubles all thresholds, leaving the tree structure identical. Feature scaling is critical for distance-based models (KNN, SVM with RBF kernel), regularised linear models (Ridge, Lasso), and neural networks. It has **zero effect** on trees. Applying it doesn't hurt, but it's wasted effort and can confuse downstream analysis of feature importances.

---

### TRICK Q5 💀 "Is OOB error always a good substitute for cross-validation?"

**Common wrong answer:** "Yes, OOB error equals cross-validation error so you can always use it."

**Correct answer:**  
OOB error is a **convenient approximation** of leave-one-out cross-validation, but not a perfect substitute. The OOB estimate averages predictions from roughly $B \cdot 0.368$ trees (those for which the sample was OOB), while the full forest uses $B$ trees — so OOB predictions are slightly pessimistically biased (lower-quality estimators used). Furthermore, OOB error only applies to **Random Forests** (not other model types), and if you need to compare RF against logistic regression or XGBoost, you need a consistent evaluation protocol — hence CV. For time-series data with temporal structure, standard OOB sampling violates the temporal ordering and would give optimistically biased estimates.

---

### TRICK Q6 💀 "Random Forest gives you feature importances — does that mean you can use them for causal inference?"

**Common wrong answer:** "Yes, high importance means the feature causes the outcome."

**Correct answer:**  
**Absolutely not.** Feature importance in RF measures **predictive association**, not causation. Correlated features can share importance scores arbitrarily (MDI especially), and a feature can have high importance purely because it is a proxy for an unobserved confounder. Permutation importance is better but still measures marginal association. Causal inference requires experimental or quasi-experimental designs, structural assumptions, and tools like DAGs / do-calculus / instrumental variables. A classic failure mode: in medical data, "days in hospital" has high RF importance for predicting mortality — but that's because sicker patients stay longer, not because hospitalisation causes death.

---

## Section 4: Advanced Questions

---

### Q23 🔴 Explain MDI (Mean Decrease in Impurity) bias toward high-cardinality features.

**Answer:**  
MDI sums the total impurity reduction contributed by a feature across all splits in all trees. High-cardinality features (e.g., a user ID with thousands of unique values, or a continuous feature) can create many distinct splits — each potentially small — but their cumulative impurity reduction accumulates to large values. A binary feature (only 2 possible values) can contribute at most one split per node, while a continuous feature can contribute at every node with a different threshold. This causes MDI to systematically overestimate the importance of high-cardinality and continuous features relative to binary or low-cardinality features. **Permutation importance** does not suffer from this bias because it directly measures the effect on held-out prediction quality, independent of feature cardinality.

---

### Q24 🔴 What is Isolation Forest, and how does it use the Random Forest idea for anomaly detection?

**Answer:**  
Isolation Forest (Liu et al., 2008) exploits the intuition that **anomalies are few and different** — they are easy to isolate with few random splits. An isolation tree recursively partitions the data by randomly selecting a feature and a random split value within the feature's range. The **path length** to isolate a point (number of splits needed) is the key statistic: anomalies require fewer splits because they lie in sparse regions of feature space and are quickly separated from the bulk. Normal points require many splits. The anomaly score is the average path length across many isolation trees, normalised by the expected path length for a dataset of that size:

$$
s(x, n) = 2^{-\frac{E[h(x)]}{c(n)}}, \quad c(n) = 2H(n-1) - \frac{2(n-1)}{n}
$$

Scores near 1 indicate anomalies; near 0.5 indicate normal points.

---

### Q25 🔴 How would you use Random Forest for feature selection?

**Answer:**  
Several approaches: (1) **Threshold on importances** — train RF on all features, discard features below an importance percentile, retrain. Risk: MDI bias can drop genuinely useful low-cardinality features. (2) **Recursive Feature Elimination (RFE)** — iteratively remove the least important feature and retrain, evaluating OOB error at each step. More reliable but $O(p)$ times slower. (3) **Boruta algorithm** — creates shadow features (random permutations of real features), runs RF, and tests whether each real feature's importance is significantly higher than the best shadow feature using a statistical test. More statistically principled. For correlated features, any importance-based method struggles — consider hierarchical clustering of features first, then one representative per cluster.

---

### Q26 🔴 What is the proximity matrix in Random Forest and what can it be used for?

**Answer:**  
The proximity matrix $P \in \mathbb{R}^{n \times n}$ stores, for every pair of training samples, the **fraction of trees in which both samples land in the same leaf node**: $P_{ij} = \frac{1}{B}\sum_b \mathbf{1}[\text{leaf}_b(x_i) = \text{leaf}_b(x_j)]$. Samples that are similar (by the RF's learned feature space) co-localise frequently. Applications: (1) **Imputing missing values** — missing feature values are replaced by a proximity-weighted average of non-missing values. (2) **Outlier detection** — samples with low average proximity to all others are outlier candidates. (3) **Visualisation** — apply MDS or t-SNE to $1 - P$ as a distance matrix to get a 2D embedding of the data in RF-learned space. (4) **Prototype finding** — cluster via the proximity matrix.

---

### Q27 🔴 How does LightGBM's leaf-wise growth differ from XGBoost's level-wise growth?

**Answer:**  
XGBoost grows trees **level-wise** (breadth-first): it splits all leaves at depth $d$ before moving to depth $d+1$, ensuring balanced trees. LightGBM grows trees **leaf-wise** (best-first): at each step it splits the single leaf with the largest loss reduction, regardless of depth. This produces asymmetric (caterpillar-shaped) trees that can achieve lower training loss with fewer total leaves. **Pros of leaf-wise:** faster convergence, often better accuracy for a given number of leaves. **Cons:** more prone to overfitting on small datasets (an asymmetric deep path can memorise sequences of samples). LightGBM adds `max_depth` and `num_leaves` constraints to mitigate this. LightGBM also uses **histogram-based splitting** (bucket continuous features) for $O(\text{bins})$ split search vs $O(n)$ in XGBoost, giving it a significant speed advantage on large datasets.

---

### Q28 🔴 Explain Bayesian hyperparameter optimisation for Random Forest. When is it better than grid search?

**Answer:**  
Bayesian optimisation models the hyperparameter-to-performance mapping as a **Gaussian Process** (or Tree Parzen Estimator), maintaining a posterior belief over which hyperparameter regions are promising. It selects the next configuration to try by maximising an **acquisition function** (e.g., Expected Improvement) that balances exploitation (try near the current best) and exploration (try uncertain regions). After each evaluation, the posterior is updated. This is more sample-efficient than grid search (which wastes evaluations on unimportant dimensions) or random search (which doesn't learn from previous evaluations). Bayesian optimisation shines when: (1) each evaluation is expensive (large dataset, slow training), (2) the search space is high-dimensional, (3) you have a small evaluation budget. Libraries: `optuna`, `hyperopt`, `scikit-optimize`, `Ax`.

---

---
## Section 5: Code Demonstrations

The following cells demonstrate the key concepts discussed above with runnable code.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import make_classification, make_regression, load_breast_cancer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, ExtraTreesClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)
print("Libraries loaded successfully.")

### Demo 1: Basic Random Forest — OOB Error vs Number of Trees

This demo shows how OOB error converges as we add more trees, illustrating the diminishing-returns property (Trick Q1).

In [ ]:
# ── Generate a moderately complex classification dataset ──
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, n_clusters_per_class=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── Track OOB error and test error as n_estimators grows ──
tree_counts = list(range(1, 201, 5))
oob_errors, test_errors = [], []

for n_trees in tree_counts:
    rf = RandomForestClassifier(
        n_estimators=n_trees,
        oob_score=True,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    oob_errors.append(1 - rf.oob_score_)
    test_errors.append(1 - accuracy_score(y_test, rf.predict(X_test)))

# ── Also compare: single deep tree vs RF ──
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)
single_tree_test_err = 1 - accuracy_score(y_test, single_tree.predict(X_test))

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(tree_counts, oob_errors, label='OOB Error', color='steelblue', linewidth=2)
axes[0].plot(tree_counts, test_errors, label='Test Error', color='darkorange', linewidth=2, linestyle='--')
axes[0].axhline(single_tree_test_err, color='crimson', linestyle=':', linewidth=2,
                label=f'Single Tree Error ({single_tree_test_err:.3f})')
axes[0].set_xlabel('Number of Trees')
axes[0].set_ylabel('Error Rate')
axes[0].set_title('OOB Error vs Test Error as n_estimators Grows')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Visualise no-extrapolation property (Trick Q3) ──
np.random.seed(0)
X_reg = np.sort(5 * np.random.rand(100, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + np.random.randn(100) * 0.1

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_reg, y_reg)

# ── Predict on extended range beyond training data ──
X_extended = np.linspace(0, 9, 500).reshape(-1, 1)
y_pred_extended = rf_reg.predict(X_extended)
y_true_extended = np.sin(X_extended).ravel()

axes[1].scatter(X_reg, y_reg, color='steelblue', s=15, label='Training data (x ∈ [0, 5])', zorder=3)
axes[1].plot(X_extended, y_pred_extended, color='darkorange', linewidth=2, label='RF prediction')
axes[1].plot(X_extended, y_true_extended, color='green', linewidth=1.5, linestyle='--', label='True sin(x)')
axes[1].axvline(5, color='crimson', linestyle=':', linewidth=2, label='Training boundary')
axes[1].fill_betweenx([-1.5, 1.5], 5, 9, alpha=0.08, color='crimson', label='Extrapolation zone')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_title('RF Cannot Extrapolate Beyond Training Range (Trick Q3)')
axes[1].legend(fontsize=8)
axes[1].set_ylim(-1.5, 1.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal OOB Error:  {oob_errors[-1]:.4f}")
print(f"Final Test Error: {test_errors[-1]:.4f}")
print(f"Single Tree Error: {single_tree_test_err:.4f}")
print("\n→ OOB error closely tracks test error (free validation estimate!)")
print("→ RF plateaus around 100-150 trees — adding more provides no benefit")
print("→ In the extrapolation zone (x > 5), RF flatlines at the last seen value")

### Demo 2: MDI vs Permutation Feature Importance — The Cardinality Bias

This demo illustrates why MDI (Gini importance) is biased toward high-cardinality features while permutation importance is not (Q7, Q23).

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.inspection import permutation_importance

# ── Load real dataset for meaningful comparison ──
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target
feature_names = data.feature_names

X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.3, random_state=42)

rf_bc = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42, n_jobs=-1)
rf_bc.fit(X_tr, y_tr)

# ── MDI importance (built-in) ──
mdi_importances = rf_bc.feature_importances_
mdi_std = np.std([tree.feature_importances_ for tree in rf_bc.estimators_], axis=0)

# ── Permutation importance on test set ──
perm_result = permutation_importance(
    rf_bc, X_te, y_te, n_repeats=20, random_state=42, n_jobs=-1
)
perm_importances = perm_result.importances_mean
perm_std = perm_result.importances_std

# ── Sort by MDI ──
mdi_order = np.argsort(mdi_importances)[::-1][:15]
perm_order = np.argsort(perm_importances)[::-1][:15]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# MDI plot
axes[0].barh(
    range(15), mdi_importances[mdi_order][::-1],
    xerr=mdi_std[mdi_order][::-1],
    color='steelblue', alpha=0.8, capsize=3
)
axes[0].set_yticks(range(15))
axes[0].set_yticklabels([feature_names[i] for i in mdi_order[::-1]], fontsize=9)
axes[0].set_xlabel('Mean Decrease in Impurity (MDI)')
axes[0].set_title('MDI Feature Importance\n(biased toward high-cardinality features)', fontsize=11)
axes[0].grid(True, alpha=0.3, axis='x')

# Permutation importance plot
axes[1].barh(
    range(15), perm_importances[perm_order][::-1],
    xerr=perm_std[perm_order][::-1],
    color='darkorange', alpha=0.8, capsize=3
)
axes[1].set_yticks(range(15))
axes[1].set_yticklabels([feature_names[i] for i in perm_order[::-1]], fontsize=9)
axes[1].set_xlabel('Mean Decrease in Accuracy (Permutation)')
axes[1].set_title('Permutation Feature Importance\n(unbiased, test-set estimate)', fontsize=11)
axes[1].grid(True, alpha=0.3, axis='x')

plt.suptitle('MDI vs Permutation Importance on Breast Cancer Dataset', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ── Rank correlation between the two methods ──
from scipy.stats import spearmanr
rho, p = spearmanr(mdi_importances, perm_importances)
print(f"Spearman rank correlation (MDI vs Permutation): ρ = {rho:.3f} (p = {p:.4f})")
print(f"OOB accuracy: {rf_bc.oob_score_:.4f}")
print(f"Test accuracy: {accuracy_score(y_te, rf_bc.predict(X_te)):.4f}")
print("\n→ High correlation here — but diverges significantly on datasets with mixed cardinality features")
print("→ Note: error bars on permutation importance reveal which features have STABLE importances")

### Demo 3: Bias-Variance Decomposition — Single Tree vs Random Forest

Empirically decomposes bias² and variance contributions by training on many bootstrap samples of the data.

In [ ]:
# ── Bias-variance decomposition via repeated experiments ──
# For a regression task, bias² = (E[ŷ] - y)², variance = E[(ŷ - E[ŷ])²]

def bias_variance_decompose(model_class, model_kwargs, X_train_full, y_train_full,
                             X_test, y_test, n_bootstrap=50, sample_size=300):
    """Empirically estimate bias² and variance via bootstrap resampling."""
    all_preds = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(len(X_train_full), size=sample_size, replace=True)
        X_b, y_b = X_train_full[idx], y_train_full[idx]
        model = model_class(**model_kwargs)
        model.fit(X_b, y_b)
        all_preds.append(model.predict(X_test))

    all_preds = np.array(all_preds)   # shape: (n_bootstrap, n_test)
    mean_pred = all_preds.mean(axis=0)

    bias_sq = np.mean((mean_pred - y_test) ** 2)
    variance = np.mean(np.var(all_preds, axis=0))
    noise = np.mean((y_test - y_test.mean()) ** 2) * 0          # irreducible noise ~ 0 here
    total_error = np.mean([(all_preds[i] - y_test) ** 2 for i in range(n_bootstrap)])

    return bias_sq, variance, total_error


np.random.seed(42)
X_bv, y_bv = make_regression(n_samples=1000, n_features=15, n_informative=8, noise=15, random_state=42)
X_tr_bv, X_te_bv, y_tr_bv, y_te_bv = train_test_split(X_bv, y_bv, test_size=0.3, random_state=42)

configs = [
    ("Decision Tree\n(depth=2)",  DecisionTreeClassifier, {"max_depth": 2}),
    ("Decision Tree\n(depth=5)",  DecisionTreeClassifier, {"max_depth": 5}),
    ("Decision Tree\n(unlimited)", DecisionTreeClassifier, {}),
    ("RF (10 trees)",             RandomForestRegressor, {"n_estimators": 10, "random_state": 42}),
    ("RF (100 trees)",            RandomForestRegressor, {"n_estimators": 100, "random_state": 42}),
    ("RF (500 trees)",            RandomForestRegressor, {"n_estimators": 500, "random_state": 42}),
]

# Use regression versions for all
from sklearn.tree import DecisionTreeRegressor
configs_reg = [
    ("DTree depth=2",    DecisionTreeRegressor, {"max_depth": 2}),
    ("DTree depth=5",    DecisionTreeRegressor, {"max_depth": 5}),
    ("DTree unlimited",  DecisionTreeRegressor, {}),
    ("RF  10 trees",     RandomForestRegressor, {"n_estimators": 10,  "random_state": 42, "n_jobs": -1}),
    ("RF 100 trees",     RandomForestRegressor, {"n_estimators": 100, "random_state": 42, "n_jobs": -1}),
    ("RF 500 trees",     RandomForestRegressor, {"n_estimators": 500, "random_state": 42, "n_jobs": -1}),
]

results = []
for label, cls, kwargs in configs_reg:
    b2, v, total = bias_variance_decompose(cls, kwargs, X_tr_bv, y_tr_bv, X_te_bv, y_te_bv, n_bootstrap=30)
    results.append((label, b2, v, total))
    print(f"{label:20s} | Bias²={b2:8.2f} | Variance={v:8.2f} | MSE≈{total:8.2f}")

# ── Plot ──
labels  = [r[0] for r in results]
bias2   = [r[1] for r in results]
varianc = [r[2] for r in results]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
bars1 = ax.bar(x - width/2, bias2,   width, label='Bias²',    color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + width/2, varianc, width, label='Variance', color='darkorange', alpha=0.85)

ax.set_xlabel('Model Configuration')
ax.set_ylabel('MSE Component')
ax.set_title('Bias² vs Variance Decomposition\n(Decision Trees vs Random Forests)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Annotate with dividing line between tree types and RF
ax.axvline(2.5, color='crimson', linestyle='--', linewidth=1.5, alpha=0.7)
ax.text(1.0, ax.get_ylim()[1]*0.92, '← Single Trees', ha='center', color='crimson', fontsize=10)
ax.text(4.0, ax.get_ylim()[1]*0.92, 'Random Forests →', ha='center', color='crimson', fontsize=10)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("• Shallow trees: HIGH bias, LOW variance")
print("• Deep trees: LOW bias, HIGH variance")
print("• RF (many trees): LOW bias (same as deep tree), MUCH LOWER variance")
print("• Adding more trees to RF: variance keeps dropping, bias stays flat")

### Demo 4: Feature Subsampling — Effect of max_features on Tree Correlation and Ensemble Performance

In [ ]:
# ── Demonstrate how max_features affects tree diversity and accuracy ──
# Lower max_features → more diverse (less correlated) trees → better ensemble despite weaker individuals

X_div, y_div = make_classification(
    n_samples=800, n_features=30, n_informative=10, n_redundant=10,
    random_state=42
)
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_div, y_div, test_size=0.25, random_state=42)

max_features_values = [1, 2, 3, 5, 7, 10, 15, 20, 30]
n_features_total = X_div.shape[1]

individual_accs = []
ensemble_accs   = []
tree_correlations = []

N_TREES = 100

for mf in max_features_values:
    rf = RandomForestClassifier(
        n_estimators=N_TREES, max_features=mf,
        oob_score=True, random_state=42, n_jobs=-1
    )
    rf.fit(X_tr_d, y_tr_d)

    # Individual tree accuracy (average over trees)
    ind_accs = [
        accuracy_score(y_te_d, tree.predict(X_te_d))
        for tree in rf.estimators_
    ]
    individual_accs.append(np.mean(ind_accs))

    # Ensemble accuracy
    ensemble_accs.append(accuracy_score(y_te_d, rf.predict(X_te_d)))

    # Pairwise tree correlation (on predictions — proxy for ρ)
    preds_matrix = np.array([tree.predict(X_te_d) for tree in rf.estimators_[:30]])
    corr_matrix  = np.corrcoef(preds_matrix)
    upper_tri    = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
    tree_correlations.append(np.mean(upper_tri))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: accuracy vs max_features
axes[0].plot(max_features_values, individual_accs, 'o--', color='steelblue',
             linewidth=2, markersize=6, label='Avg individual tree accuracy')
axes[0].plot(max_features_values, ensemble_accs,   'o-',  color='darkorange',
             linewidth=2, markersize=6, label='Ensemble accuracy')
axes[0].axvline(int(np.sqrt(n_features_total)), color='crimson', linestyle=':',
                linewidth=2, label=f'Default √p = {int(np.sqrt(n_features_total))}')
axes[0].set_xlabel('max_features (m)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Individual Tree vs Ensemble Accuracy\nvs Feature Subsampling Size')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2: tree correlation vs max_features
axes[1].plot(max_features_values, tree_correlations, 's-', color='purple',
             linewidth=2, markersize=6)
axes[1].axvline(int(np.sqrt(n_features_total)), color='crimson', linestyle=':',
                linewidth=2, label=f'Default √p = {int(np.sqrt(n_features_total))}')
axes[1].set_xlabel('max_features (m)')
axes[1].set_ylabel('Mean pairwise tree correlation (ρ)')
axes[1].set_title('Inter-Tree Correlation vs Feature Subsampling\n(lower ρ → more variance reduction)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Effect of max_features on RF Diversity and Performance (p={n_features_total} features)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Key observations:")
print(f"• m=1 (max diversity):     individual acc={individual_accs[0]:.3f}, ensemble acc={ensemble_accs[0]:.3f}, ρ={tree_correlations[0]:.3f}")
print(f"• m=√p (default):          individual acc={individual_accs[max_features_values.index(int(np.sqrt(n_features_total)))]:.3f}, "
      f"ensemble acc={ensemble_accs[max_features_values.index(int(np.sqrt(n_features_total)))]:.3f}, "
      f"ρ={tree_correlations[max_features_values.index(int(np.sqrt(n_features_total)))]:.3f}")
print(f"• m=p (plain bagging):     individual acc={individual_accs[-1]:.3f}, ensemble acc={ensemble_accs[-1]:.3f}, ρ={tree_correlations[-1]:.3f}")
print("\n→ As m increases: individual trees get stronger BUT trees get more correlated")
print("→ The √p default strikes the empirically optimal balance between strength and diversity")

---
## Section 6: Quick Reference Summary

### Core Formulas

| Concept | Formula / Rule |
|---|---|
| Ensemble variance | $\text{Var}(\bar{T}) = \rho\sigma^2 + \frac{(1-\rho)\sigma^2}{B}$ |
| OOB probability | $P(\text{not selected}) \to e^{-1} \approx 36.8\%$ |
| Default m (classification) | $m = \lfloor\sqrt{p}\rfloor$ |
| Default m (regression) | $m = \lfloor p/3 \rfloor$ |
| Gini impurity | $1 - \sum_k p_k^2$ |
| Entropy | $-\sum_k p_k \log_2 p_k$ |
| AdaBoost learner weight | $\alpha_t = \frac{1}{2}\ln\frac{1-\epsilon_t}{\epsilon_t}$ |
| GBM pseudo-residuals | $r_{im} = -\partial L / \partial F(x_i)$ |

---

### What Random Forest Can and Cannot Do

| ✅ CAN | ❌ CANNOT |
|---|---|
| Handle mixed feature types | Extrapolate beyond training range |
| Handle high-dimensional data | Capture linear relationships efficiently |
| Provide feature importance | Provide causal importance |
| Estimate generalisation via OOB | Replace CV for cross-model comparison |
| Handle missing values (with proximity imputation) | Natively handle missings in sklearn |
| Scale to large datasets (parallelisable) | Train incrementally (must retrain) |
| Handle class imbalance (with weighting) | Guarantee calibrated probabilities |

---

### Ensemble Method Comparison

| Method | Base Learner | Training | Targets | Overfit Risk | Speed |
|---|---|---|---|---|---|
| Bagging | Any (usually deep tree) | Parallel | Variance ↓ | Low | Fast |
| Random Forest | Deep tree | Parallel | Variance ↓ | Low | Fast |
| Extra Trees | Deep tree | Parallel | Variance ↓↓ | Low | Fastest |
| AdaBoost | Shallow tree (stump) | Sequential | Bias ↓ | Medium | Medium |
| Gradient Boosting | Shallow tree | Sequential | Bias ↓ + Variance ↓ | Medium-High | Slow |
| XGBoost / LightGBM | Shallow tree | Sequential | Bias ↓ + Variance ↓ | Medium | Medium-Fast |
| Stacking | Any (diverse) | Parallel + 1 meta | Both | Low-Medium | Depends |

---

### Top Interview Gotchas — Instant Recall

1. **More trees always help?** → Diminish and plateau, never overfit, but cost keeps growing.
2. **Feature scaling?** → Irrelevant for trees. Don't waste time scaling for RF.
3. **Can RF extrapolate?** → **No.** Hard limit: predictions are clamped to training range.
4. **MDI vs permutation?** → MDI is biased toward high cardinality. Use permutation for interpretation.
5. **Feature importance = causation?** → Never. Importance = predictive association.
6. **OOB ≡ cross-validation?** → Approximate, not exact. Slightly pessimistic; can't compare to other models.
7. **Random Forest linear or non-linear?** → Non-linear globally, but linear in leaf-indicator space.
8. **Bagging reduces bias?** → No! Only variance. Bias stays the same.
9. **Boosting reduces variance?** → Yes (averaging helps), but its primary goal is bias reduction.
10. **Key hyperparameter to tune?** → `max_features` first, then `max_depth`, then `n_estimators`.

---

*This notebook is part of the **ML Core Concepts** series. See also: `01_decision_trees_interview_qa.ipynb`.*